#Basic Tasks

In [0]:
%sql
--1
create table cyntexa_dev.sales.customers1 as
select * from read_files('/Volumes/dev/bronze/raw/customers/customers_raw.csv')

In [0]:
%sql
--2
create or replace table cyntexa_dev.sales.products1 as
select
  p.id,
  p.name,
  p.category,
  p.price.amount as price_amount,    
  p.price.currency as price_currency   
from read_files('/Volumes/cyntexa_dev/sales/raw/products_simple.json', format => 'json', multiline => true) t
lateral view explode(t.products) exploded_table as p

In [0]:
%sql
--3
describe cyntexa_dev.sales.customers1;
describe extended cyntexa_dev.sales.customers1;
describe detail cyntexa_dev.sales.customers1;

In [0]:
%sql
describe cyntexa_dev.sales.products1;
describe extended cyntexa_dev.sales.products1;
describe detail cyntexa_dev.sales.products1;

3. note

Describe → gives columns only

describe extended → gives columns + general table metadata(like owner, created time, location)

describe detail → gives delta storage internals (size, files, format version)

#Intermediate Tasks

In [0]:
%sql
--4
create or replace table cyntexa_dev.sales.customers1 as
select *, _metadata.file_name, _metadata.file_path from read_files('/Volumes/dev/bronze/raw/customers/customers_raw.csv')

In [0]:
%sql
create or replace table cyntexa_dev.sales.products1 as
select
  p.id,
  p.name,
  p.category,
  p.price.amount      as price_amount,    
  p.price.currency    as price_currency,  
  _metadata.file_name as source_file_name,
  _metadata.file_path as source_file_path
from read_files(
  '/Volumes/cyntexa_dev/sales/raw/products_simple.json',
  format => 'json',
  multiline => true
) t
lateral view explode(t.products) exploded_table as p

In [0]:
%sql
select * from cyntexa_dev.sales.customers1 limit 5

In [0]:
%sql
select * from cyntexa_dev.sales.products1;

In [0]:
%sql
--5
create table cyntexa_dev.sales.customers1_iceberg
using iceberg as
select *, _metadata.file_name, _metadata.file_path from read_files('/Volumes/dev/bronze/raw/customers/customers_raw.csv')

In [0]:
%sql
describe detail cyntexa_dev.sales.customers1_iceberg

describe detail clearly shows format field as delta vs iceberg, and the location path structure differs because of the different metadata folder each one creates

In [0]:
%sql
--6
select _metadata.file_name, count(*) as record_count from cyntexa_dev.sales.customers1 group by  _metadata.file_name order by  _metadata.file_name

#Advanced Tasks

In [0]:
%sql
--7
select * from read_files('/Volumes/cyntexa_dev/sales/raw/customers_20260901.csv', format => 'csv', header => true) 

In [0]:
%sql
select * from read_files('/Volumes/cyntexa_dev/sales/raw/customers_20260902.csv', format => 'csv', header => true, delimiter => ';') limit 5

In [0]:
%sql
select * from read_files(
  '/Volumes/cyntexa_dev/sales/raw/customers_20260903.csv', format => 'csv', header => true) 

--8

for cyntexa we can use delta as the default table format because it works best inside Databricks and has all the features supported without extra setup but if some other tool like snowflake or trino needs to read our table directly then plain Delta becomes a problem since those tools understand iceberg format better
So in that case we should either make a native Iceberg table or turn on delta UniForm, which lets the same table be read both as Delta and Iceberg together. Basically: use Delta normally, switch only when outside tools actually need access.

In [0]:
%sql
--9
describe history cyntexa_dev.sales.customers1